In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from fastai.vision.all import *
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
    HiResCAM,
    ScoreCAM,
    GradCAMPlusPlus,
    AblationCAM,
    XGradCAM,
    EigenCAM,
    FullGrad,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPadDataset, BlurPad4ChanDataset
from mtrain.neg_mask.model.show import show_confusion_matrix_using_preds
from mtrain.neg_mask.model.show import show_classification_report
from mtrain.neg_mask.model.show import get_preds_for_ds

from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain



In [ ]:
NUM_CHAN = 3

def get_denormalized(tens):
    image, mask = None, None
    if NUM_CHAN == 4:
        image, mask = denormalize_4chan_imagenet(tens)
    else:
        image = denormalize_imagenet(tens)
    
    image = image.permute([1,2,0]).numpy()
    if mask is not None:
        mask = mask.numpy()
    return image, mask
    

def show_gradcam_for_image(learn, input_tensor, target_label_idx=None):
    target_layers = [learn.model.get_submodule("0.7.1.conv1")]
    img_arr, _ = get_denormalized(input_tensor[0])
    # combined = torch.cat([denorm_image, denorm_mask])
    # print(combined.shape)


    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(
            img_arr, grayscale_cam, use_rgb=True
        )
        model_outputs = cam.outputs

        return visualization, img_arr

def get_original_image(dataset_path):
    CLEAN = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean")
    path_name = Path(dataset_path).name
    label_part = path_name.split("_")[0]
    other_part = Path(path_name[len(label_part) + 1: ])

    image_path = CLEAN / label_part / other_part.stem / "orig.jpg"
    return image_path

def get_0pad_path(dataset_path):
    B5P5_NOISY = Path(
        '/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy'
    )
    path_name = Path(dataset_path).name
    return B5P5_NOISY / "train" / path_name

def get_0pad_learner():
    image_paths = list((B5P5_NOISY / "train").glob("*.jpg"))
    stratify = [BlurPadDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )
    train_ds = BlurPadDataset(train_paths, DS_PATH / "masks", 130, False, max_noise=30)
    valid_ds = BlurPadDataset(valid_paths, DS_PATH / "masks", 130, True, max_noise=None)
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
        persistent_workers=True,
    )  # don't respawn workers each epoch)
    state_dict = torch.load("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/0pad/fullset-iter30.pt", map_location=default_device())
    learner = get_learner(dls)
    learner.model.load_state_dict(state_dict, strict=True)
    return learner

def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)

# 0pad train and unblurred train

- 0pad training is done
  - the model looks okay, i could be happier but eh
  - its focus seems fine, im continuing with it
  - one interesting thing is that it seems to focus on the "center" for trash when it is trash, and vice versa, instead of the focus being at that same place everytime, this is something worth investigating, why is gradcam behaving the way it is

## dataset

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPadGaussianDataset
from sklearn.model_selection import train_test_split

LABELS = ["other", "trash"]
IS_BLUR = True


B5P5_NOISY = Path(
    '/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy'
)
B5P5_BLUR_K13S4 = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_blur_k13s4')

B5P10_BLUR_K13S4 = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p10_blur_k13s4')
B5P10_K3S1 = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p10_blur_k3s1')

DS_PATH = B5P10_K3S1 if IS_BLUR else B5P5_NOISY

NUM_CHAN = 3

DatasetClass = BlurPadGaussianDataset if NUM_CHAN == 4 else BlurPadDataset
DatasetClass = BlurPadGaussianDataset
# DatasetClass = BlurPadDataset

image_paths = list((DS_PATH / "train").glob("*.jpg"))
stratify = [DatasetClass.label_func(p) for p in image_paths]
train_paths, valid_paths = train_test_split(
    image_paths, test_size=0.2, stratify=stratify, random_state=42
)

In [ ]:
# tlen, vlen = len(train_paths), len(valid_paths)
# print(tlen, vlen)
# # overfit on less data first, to make it look at the fucking center
# train_paths = train_paths[:300]
# valid_paths = valid_paths[:50]

In [ ]:
# should_noise = not IS_BLUR
# noise = 30 if should_noise else None

# force use noise now to make the smaller kernels jitter
noise = 30
print("using noise", noise)

train_ds = DatasetClass(train_paths, DS_PATH / "masks", 130, False, max_noise=noise)
valid_ds = DatasetClass(valid_paths, DS_PATH / "masks", 130, True, max_noise=None)

In [ ]:
dls = DataLoaders.from_dsets(
    train_ds,
    valid_ds,
    device=default_device(),
    num_workers=4,
    bs=16,
    # pin_memory=True,
    persistent_workers=True,
)  # don't respawn workers each epoch)

In [ ]:
idx = 9
ds = train_ds
tens, targ = ds[idx]
print("target", targ)
print("shape", tens.shape)
img, _ = get_denormalized(tens)
# orig_img = plt.imread(get_original_image(ds.image_paths[idx]))
plt.imshow(img, cmap="gray")
# show([img], ncols=1)
# print(mask)
# print(img)
# print(mask.shape)
# show([img, mask],nols=2, cmap="gray")k

In [ ]:
CLS_WEIGHT=torch.tensor([1.0, 2.5]).float().to("mps")

def get_learner(dls):
    learn = vision_learner(
        dls,
        resnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=NUM_CHAN,
    )
    learn = learn.remove_cb(ProgressCallback)
    return learn

In [ ]:
def get_state_dict(path):
    # state_dict = torch.load("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt", map_location=default_device())
    state_dict = torch.load(path, map_location=default_device())

    new_sd = state_dict.copy()

    old_weights = state_dict["0.0.weight"]
    num_in_kernels = old_weights.shape[1]
    if NUM_CHAN == 4 and num_in_kernels == 3:
        print("channel mismatch, inisitalising channel 4 with mean weights and returning dict")
        # new_channel = old_weights.mean(dim=1, keepdim=True)
        new_channel, _ = old_weights.max(dim=1, keepdim=True)
        # print(max_chan.shape, new_channel.shape)
        new_weights = torch.cat([old_weights, new_channel], dim=1)
        new_sd["0.0.weight"] = new_weights
    return new_sd


# MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt"
MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/clean-with-subtract/fullset-p10-iter5.pt"
learner = get_learner(dls)
old_weights = get_state_dict(MODEL_PATH)
learner.model.load_state_dict(old_weights, strict=True)

In [ ]:
from torch import nn

class AdaptiveMultiScale(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv7 = nn.Conv2d(in_channels, out_channels, 7, stride=2, padding=3, bias=False)
        self.conv15 = nn.Conv2d(in_channels, out_channels, 15, stride=2, padding=7, bias=False)
        
        # A small 'Gate' that looks at the input and decides which kernel to trust
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        g = self.gate(x)
        # If noise is high, the gate learns to multiply conv7 by a low value
        return (self.conv7(x) * g) + (self.conv15(x) * (1 - g))

In [ ]:
learner.model[0][0] = AdaptiveMultiScale(3, 64)
# Assuming 'old_weights' is your [64, 3, 7, 7] tensor from the original state_dict
# and 'model' is your ResNet with the new MultiScaleFirstLayer at model[0][0]

with torch.no_grad():
    # --- Initialize Conv7 (The Local Path) ---
    # We copy your pre-trained weights directly here
    learner.model[0][0].conv7.weight.copy_(old_weights["0.0.weight"])
    
    # --- Initialize Conv15 (The Global Path) ---
    # Since this is a brand new scale, we use Kaiming initialization
    nn.init.kaiming_normal_(learner.model[0][0].conv15.weight, mode='fan_out', nonlinearity='relu')
    
    # --- Initialize the Gate ---
    # We want the model to start by trusting the 7x7 weights it already knows.
    # By zeroing the weights and biases of the gate's conv layer, 
    # the Sigmoid output starts at 0.5 (an even blend).
    nn.init.zeros_(learner.model[0][0].gate[1].weight)
    nn.init.zeros_(learner.model[0][0].gate[1].bias)

print("Weights successfully grafted!")

In [ ]:
learner.unfreeze()
# train the initial layers first
learner.fit_one_cycle(10, lr_max=slice(1e-3, 1e-6))

In [ ]:
# learner.freeze()
# # only lasy layer
# for param in learner.model.get_submodule("0.0").parameters():
#     param.requires_grad = False

# for param in learner.model.get_submodule("1.8").parameters():
#     param.requires_grad = True

# learner.fine_tune(1)

In [ ]:
learner.lr_find()

In [ ]:
learner.fit_one_cycle(5, lr_max=slice(1e-5, 1e-4))

In [ ]:
torch.save(learner.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/clean-with-subtract/fullset-p10-iter5.pt")

In [ ]:
learner.fit_one_cycle(100, lr_max=slice(1e-7,1e-6))

the model really has problems i feel now lol. I'll decrease the size of dls now to overfit on some examples to see if it is focusing on the correct parts

In [ ]:
import torch
torch.save(learner.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt")

In [ ]:
import torch
torch.save(learner.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-iter10.pt")

In [ ]:
learner.eval()

In [ ]:
preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
probs, targs, decoded, losses = preds
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
i = 0

In [ ]:

i, (decoded != targs).sum()

In [ ]:
i = 0

In [ ]:
# i-=1
idx = top_loss_idxs[i]
predicted = learner.predict(learner.dls.valid_ds[idx])[0]
viz_trash, img = show_gradcam_for_image(learner, learner.dls.valid_ds[idx][0].unsqueeze(0), 1)
viz_other, _ = show_gradcam_for_image(learner, learner.dls.valid_ds[idx][0].unsqueeze(0), 0)


crop_path = learner.dls.valid_ds.image_paths[idx]
# img0pad = plt.imread(get_0pad_path(crop_path))
# l0_idx = path_by_idx_l_0pad[crop_path.name]
# l0_pred = learner_0pad.predict(learner_0pad.dls.valid_ds[l0_idx])[0]
# l0_viz_trash, _ = show_gradcam_for_image(learner_0pad, learner_0pad.dls.valid_ds[l0_idx][0].unsqueeze(0), 0)

orig_img = plt.imread(get_original_image(crop_path))
print("name", crop_path.name)
print("label", learner.dls.valid_ds[idx][1])
print("predicted", predicted)
# print("l0_pred", l0_pred)
show([viz_trash, viz_other, img, orig_img], (20,20), ncols=4, axis="off")
i += 1

In [ ]:
show_reports(learner)

In [ ]:
labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
show_classification_report(probs, targs, labels)
show_confusion_matrix_using_preds(probs, targs, labels)

# Old code

In [ ]:
from torchvision.transforms import v2
from fastai.vision.all import *
import albumentations as A

LOG_ROOT = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification"
)
B5P5K5S3 = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_blur_k5s3"
)


def label_func(x):
    if x.startswith("other"):
        return "other"
    elif x.startswith("trash"):
        return "trash"
    else:
        raise Exception(f"bad file name {x}")

def get_mask_where_no_noise_should_be_added(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    ret, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)

    x0, w0, y0, h0 = None, None, None, None
    for i in range(1, num_labels):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]
        
        # Filter out tiny noise if needed
        if area > 10: 
            cv2.rectangle(gray, (x, y), (x + w, y + h), (255), 2)
            x0, y0, w0, h0 = x, y, w, h
            break

    if x0 is None or y0 is None or w0 is None or h0 is None:
        # empty mask
        return np.zeros(img_rgb.shape[:2], dtype=bool)
    
    res_mask = np.zeros(img_rgb.shape[:2], dtype=bool)
    res_mask[y0:y0+h0, x0:x0+w0] = True
    return res_mask


def get_noisy_image(img_arr, noise_max):
    mask = get_mask_where_no_noise_should_be_added(img_arr)
    noisy = np.random.randint(0, noise_max, img_arr.shape, dtype=np.uint8)
    noisy[mask] = img_arr[mask]
    return noisy

# def get_crop_around_image(img_arr):
#     pass


class MyTransform(Transform):
    def encodes(self, x: PILImage):
        print("yoooooooooooo")
        # print(f"Filename: {x.name}")
        return x  # pass through, or do something with it

class ReNoisePadding(Transform):
    def __init__(self, noise_max=80):
        self.noise_max = noise_max

    def encodes(self, img: PILImage):
        arr = np.array(img)
        return PILImage.create(get_noisy_image(arr, self.noise_max))
    

# min_scale=1.0
def get_0pad_dls(data_root, log_root):
  return ImageDataLoaders.from_name_func(
      log_root,
      get_image_files(data_root),
      valid_pct=0.2,
      seed=42,
      label_func=label_func,
      item_tfms=[MyTransform(), ReNoisePadding(), CropPad(130)],
      batch_tfms=aug_transforms(max_zoom=1.0, min_scale=1.0),
      bs=4,
  )

def get_unblur_dls(data_root, log_root):
  return ImageDataLoaders.from_name_func(
      log_root,
      get_image_files(data_root),
      valid_pct=0.2,
      seed=42,
      label_func=label_func,
      item_tfms=[CropPad(130)],
      batch_tfms=aug_transforms(max_zoom=1.0, min_scale=1.0),
      # batch_tfms=[],
      bs=4,
  )

def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    # Clone the tensor to avoid modifying the original
    t = tensor.detach().clone().cpu()

    # Reshape mean and std to (3, 1, 1) to match the tensor (C, H, W)
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)

    # Apply the reverse formula
    t = (t * std) + mean

    # Optional: Clip values to [0, 1] for visualization
    return torch.clamp(t, 0, 1)

def warmup_model(learn, device):
    # bn layers are drifted, when loading the model from elsewhere, update stats
    # Put the model in TRAIN mode but don't update weights (only update BN stats)
    learn.model.train() 
    learn.model.to(device)
    total = len(learn.dls.valid)
    for i, b in tqdm(enumerate(learn.dls.valid), total=total):
        _ = learn.model(b[0])

    # Now try show_results again
    learn.model.eval()

def get_learner(dls):
    learn = vision_learner(
        dls,
        resnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(weight=torch.tensor([1.41, 3.43])),
    )
    learn = learn.remove_cb(ProgressCallback)
    return learn


In [ ]:
stage0_dls = get_0pad_dls(B5P5_NOISY, LOG_ROOT)
stage0_dls.show_batch()

In [ ]:
from torch import nn

dls = get_unblur_dls(B5P5K5S3, LOG_ROOT)
# learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage1-iter15.pkl")
# learn.dls = dls
learn = get_learner(dls)
model_path = "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/unblurred/stage0/models/unblurred-stage1-iter15"
learn = learn.load(model_path)
# warmup_model(learn, default_device())
print(len(dls.train), len(dls.valid))

stage0_dls = get_0pad_dls(B5P5_NOISY, LOG_ROOT)
# stage0_learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage0-iter20.pkl")
# stage0_learn.dls = stage0_dls

stage0_learn = get_learner(stage0_dls)
stage0_learn = stage0_learn.load(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/unblurred/stage0/models/unblurred-stage0-iter20"
)
# warmup_model(stage0_learn, default_device())
print(len(stage0_dls.train), len(stage0_dls.valid))



In [ ]:
stage0_dls.show_batch()

In [ ]:
image = stage0_dls.valid_ds.items[0]
pilimage = PILImage.create(image)
image = plt.imread(image)
print(image.shape)
lgsz = A.LongestMaxSize(130)
rsz = lgsz(image=image)["image"]
print(rsz.shape)
show([image, rsz])

In [ ]:
dir(pilimage.im)

In [ ]:
stage0_dls.show_batch()

In [ ]:
import torch
import fastcore

model_path = "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/unblurred/stage0/models/unblurred-stage1-iter15.pth"
torch.serialization.add_safe_globals([fastcore.foundation.L])
state = torch.load(model_path, map_location="cpu", weights_only=True)
hasopt = set(state)=={'model', 'opt'}
print(hasopt)

# model_state = state['model'] if hasopt else state
# get_model(model).load_state_dict(model_state, strict=strict)

In [ ]:
learn = get_learner(dls)

In [ ]:
learn.model

In [ ]:
len(stage0_dls.train_ds)

In [ ]:
import hashlib

def get_model_fingerprint(model):
    hash_m = hashlib.sha256()
    for name, param in model.state_dict().items():
        # Hash the parameter name and its data
        hash_m.update(name.encode())
        hash_m.update(param.cpu().numpy().tobytes())
    return hash_m.hexdigest()

def compare_models(model_a, model_b):
    sd_a = model_a.state_dict()
    sd_b = model_b.state_dict()
    
    mismatches = []
    for key in sd_a:
        if not torch.equal(sd_a[key], sd_b[key]):
            diff = (sd_a[key].float() - sd_b[key].float()).abs().mean().item()
            sdak , sdab = sd_a[key].float().mean(), sd_b[key].float().mean()
            # print(sd_a[key])
            mismatches.append((key, diff, sdak, sdab))
    
    if not mismatches:
        print("✅ Models are byte-for-byte identical.")
    else:
        print(f"❌ Found {len(mismatches)} mismatched layers.")
        for name, diff, sdak, sdab in mismatches[:10]: # Show first 10
            print(f"   - {name} (Avg Diff: {diff:.10f}) val={sdak} {sdab}")

In [ ]:
test_learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage1-iter15.pkl")
print("Before Export:", learn.model[0][1].running_mean[:5])
print("After export", test_learn.model[0][1].running_mean[:5])
test_stage0_learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage0-iter20.pkl")
test_stage0_learn.dls = stage0_dls
print("Before Export:", stage0_learn.model[0][1].running_mean[:5])
print("After export", test_stage0_learn.model[0][1].running_mean[:5])

In [ ]:
# Get fingerprint of the current (working) model
print(f"Current Model Hash: {get_model_fingerprint(learn.model)}")
print(f"Current Model Hash: {get_model_fingerprint(test_learn.model)}")
compare_models(learn.model.cpu(), test_learn.model.cpu())

In [ ]:
# stage0_learn.export("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage0-iter20.pkl")
# learn.export("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage1-iter15.pkl")

In [ ]:
interp = ClassificationInterpretation.from_learner(test_stage0_learn)
interp.plot_confusion_matrix()

In [ ]:
interp = ClassificationInterpretation.from_learner(stage0_learn)
interp.plot_confusion_matrix()

In [ ]:

# stage0_learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage0-iter20.pkl")
# stage0_learn.dls = stage0_dls

In [ ]:
warmup_model(learn, "mps")

In [ ]:
learn.eval()
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
learn.eval()
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
# Put the model in TRAIN mode but don't update weights (only update BN stats)
stage0_learn.model.train() 
stage0_learn.model.to("mps")
for i, b in enumerate(stage0_learn.dls.valid):
    if i > 40:
        break # Just a few batches
    _ = stage0_learn.model(b[0])

# Now try show_results again
stage0_learn.model.eval()
stage0_learn.show_results()

In [ ]:
stage0_learn.show_results()

In [ ]:
x, y = stage0_dls.valid.one_batch()
print(f"Mean: {x.mean()}, Std: {x.std()}")
print(f"Min/Max: {x.min()}, {x.max()}")

In [ ]:
stage0_learn.show_results()

In [ ]:
learn.freeze()

In [ ]:
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(5)

In [ ]:
learn.save(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/unblurred-stage1-iter15"
)

In [ ]:
interp = ClassificationInterpretation.from_learner(stage0_learn)
interp.plot_confusion_matrix()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
_, idxs = interp.top_losses(100)

In [ ]:
paths = []
for idx in idxs:
    paths.append(learn.dls.valid_ds.items[idx])

In [ ]:
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    # Clone the tensor to avoid modifying the original
    t = tensor.detach().clone().cpu()

    # Reshape mean and std to (3, 1, 1) to match the tensor (C, H, W)
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)

    # Apply the reverse formula
    t = (t * std) + mean

    # Optional: Clip values to [0, 1] for visualization
    return torch.clamp(t, 0, 1)


def show_gradcam_for_image(learn, img_path, target_label_idx=None):
    target_layers = [learn.model.get_submodule("0.7.1.conv1")]
    dl = learn.dls.test_dl([img_path])
    input_tensor = dl.one_batch()[0]
    mean, std = imagenet_stats
    denormalized = denormalize(input_tensor[0])
    img_arr = denormalized.permute([1, 2, 0])

    targets = [ClassifierOutputTarget(1)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        # transform = A.CenterCrop(height=130, width=130)

        # img = plt.imread(img_path)
        # crop = transform(image=img)["image"]
        # crop = crop.astype(np.float32) / 255

        # print(img_arr.max(), img_arr.min())
        visualization = show_cam_on_image(
            img_arr.cpu().numpy(), grayscale_cam, use_rgb=True
        )
        model_outputs = cam.outputs

        return visualization, img_arr

In [ ]:
CLEAN_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean"
)
STAGE0_IP_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy"
)


def get_originals(path):
    comps = Path(path).stem.split("_")
    label = comps[0]
    name = path.stem[len(label) + 1 :]

    d = CLEAN_DIR / label / name
    stage0_img = STAGE0_IP_DIR / label / path.name
    return d, stage0_img


def get_padded_bbox_mask(mask, padding=10):
    # 1. Find the coordinates of all non-zero pixels
    coords = cv2.findNonZero(mask)
    if coords is None:
        return np.zeros_like(mask), None

    # 2. Get the standard bounding box
    x, y, w, h = cv2.boundingRect(coords)
    img_h, img_w = mask.shape[:2]

    # 3. Apply padding with boundary constraints
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(img_w, x + w + padding)
    y2 = min(img_h, y + h + padding)

    # 4. Create the new mask
    padded_mask = np.zeros_like(mask)
    cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

    return padded_mask, (x1, y1, x2, y2)


def get_blurred_artifacts(
    img_path, mask_path, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    img = cv2.imread(img_path)
    assert img is not None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = DiskBooleanMask.load(mask_path)
    new_mask, box = get_padded_bbox_mask(mask, bbox_pad)

    blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
    only_mask_unblurred = blurred.copy()

    new_mask = new_mask.astype(bool)
    only_mask_unblurred[new_mask] = img[new_mask]

    if box is not None:
        x1, y1, x2, y2 = box
        original_crop = img[y1:y2, x1:x2]
    else:
        original_crop = None

    padded_crop = np.random.randint(
        0, 30, img.shape, dtype=np.uint8
    )  # low noise, adjust 15 to taste
    padded_crop[new_mask] = img[new_mask]

    return {
        "mask": mask,
        "padded_mask": new_mask,
        "blurred": blurred,
        "unblurred": only_mask_unblurred,
        "img": img,
        "original_crop": original_crop,
        "padded_crop": padded_crop,
    }


In [ ]:
def evaluate_path(path):
    d, s0_path = get_originals(path)
    # arts = get_blurred_artifacts(d / "orig.jpg", d / "mask.png", 3, 5, 5)
    targ = d.parent.name

    # stage0_path =
    stage0_res = stage0_learn.predict(s0_path)[0]
    stage1_res = learn.predict(path)[0]

    s1_viz, _ = show_gradcam_for_image(learn, path)
    s0_viz, _ = show_gradcam_for_image(stage0_learn, s0_path)
    print("stage0:", stage0_res)
    print("stage1", stage1_res)
    print("targ:", targ)

    img = plt.imread(d / "orig.jpg")
    s0_img = plt.imread(s0_path)
    s1_img = plt.imread(path)

    show(
        [
            s1_img,
            s1_viz,
            s0_img,
            s0_viz,
            # viz, img, s0_img, s1_img
        ],
        (20, 20),
        4,
        "off",
    )

In [ ]:
learn.eval()
stage0_learn.eval()
evaluate_path(paths[10])

In [ ]:
# def toggle_object_placement(path):
#     path = Path(path)
#     d, _ = get_originals(path)
#     label = d.parent.name
#     if label == "trash":
#         targ = "other"
#     else:
#         targ = "trash"
#     print(path)
#     our_dir = path.parent.parent

#     print("move", label, "->", targ)



In [ ]:
# toggle_object_placement(paths[10])

In [ ]:
paths[10]

In [ ]:
idx = 0
viz, img = show_gradcam_for_image(learn, paths[idx])
print(paths[idx])
item, targ = learn.dls.valid_ds[idx]

label = learn.predict(paths[idx])[0]
print(label, targ)
show([img, viz], (20, 20))

In [ ]:
from mtrain.neg_mask.crops import get_region_crops


def get_padded_bbox_mask(shape, bbox, padding=5):
    # 2. Get the standard bounding box
    img_h, img_w = shape[:2]

    # 3. Apply padding with boundary constraints
    x1 = max(0, bbox.x - padding)
    y1 = max(0, bbox.y - padding)
    x2 = min(img_w, bbox.x2 + padding)
    y2 = min(img_h, bbox.y2 + padding)

    # 4. Create the new mask
    padded_mask = np.zeros(shape, dtype=np.uint8)
    cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

    return padded_mask, (x1, y1, x2, y2)


# # to use: B5P5K5S3
# def get_blurred_artifacts(img, bbox, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5):
#     assert img is not None
#     new_mask, box = get_padded_bbox_mask(img.shape, bbox, bbox_pad)

#     blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
#     only_mask_unblurred = blurred.copy()

#     new_mask = new_mask.astype(bool)
#     only_mask_unblurred[new_mask] = img[new_mask]

#     return {
#         "padded_mask": new_mask,
#         "unblurred": only_mask_unblurred,
#         "img": img,
#     }

def get_blurred_artifacts(img, bbox, crop_size, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5):
    assert img is not None
    new_mask, box = get_padded_bbox_mask(img.shape, bbox, bbox_pad)
    blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
    only_mask_unblurred = blurred.copy()
    new_mask = new_mask.astype(bool)
    only_mask_unblurred[new_mask] = img[new_mask]

    result = {
        "padded_mask": new_mask,
        "unblurred": only_mask_unblurred,
        "img": img,
    }

    img_h, img_w = img.shape[:2]
    cx = (bbox.x + bbox.x2) // 2
    cy = (bbox.y + bbox.y2) // 2
    half = crop_size // 2

    x1 = max(0, cx - half)
    y1 = max(0, cy - half)
    x2 = min(img_w, x1 + crop_size)
    y2 = min(img_h, y1 + crop_size)
    # shift back if clamped on the far edge
    x1 = max(0, x2 - crop_size)
    y1 = max(0, y2 - crop_size)

    result["crop"] = only_mask_unblurred[y1:y2, x1:x2]
    result["crop_box"] = (x1, y1, x2, y2)

    return result

def predict_and_return_prob_masks(
    image, mask, learner, device=None, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    bboxes = list(get_region_crops(mask))
    if not bboxes:
        zero_mask = np.zeros(mask.shape, dtype=np.float32)
        return zero_mask, zero_mask

    unblurred = []
    for bbox in bboxes:
        artifacts = get_blurred_artifacts(
            image, bbox, 130, blur_kernel_sz, blur_sigma, bbox_pad
        )
        unblurred.append(artifacts["crop"])
    return unblurred

    # test_dl = learner.dls.test_dl(unblurred)
    # preds, _, decoded = learner.get_preds(dl=test_dl, with_decoded=True)
    # print(decoded)


In [ ]:
from mtrain.neg_mask.crops import Bbox
def predict_and_return_prob_masks(
    image, mask, learner, crop_size=130, device=None, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    bboxes = list(get_region_crops(mask))
    if not bboxes:
        zero_mask = np.zeros(mask.shape, dtype=np.float32)
        return zero_mask, zero_mask

    unblurred = []
    for bbox in bboxes:
        crop, new_bbox = get_centered_crop_with_coords(image, bbox, crop_size)
        artifacts = get_blurred_artifacts(image, new_bbox, blur_kernel_sz, blur_sigma, bbox_pad)
        unblurred.append(artifacts["unblurred"])
    return crop
    
    # test_dl = learner.dls.test_dl(unblurred)
    # preds, _, decoded = learner.get_preds(dl=test_dl, with_decoded=True)
    # print(decoded)

    # input_tensor = dl.one_batch()[0]
    # mean, std = imagenet_stats


    # mask = mask.astype(bool)
    # crop_data_list = [(image, mask, bbox) for bbox in bboxes]
    # probs = run_inference(learner, crop_data_list, crop_size, device)
    # prob_masks = core.reconstruct_probability_masks(image, mask, probs, bboxes)

    # other_mask, trash_mask = prob_masks[0], prob_masks[1]
    # return other_mask, trash_mask

def get_centered_crop_with_coords(array, bbox, crop_size=130):
    x, x2, y, y2 = bbox.x, bbox.x2, bbox.y, bbox.y2
    
    # 1. Find the center and the start of the crop window
    center_x, center_y = (x + x2) // 2, (y + y2) // 2
    half_size = crop_size // 2
    
    start_x = center_x - half_size
    start_y = center_y - half_size
    
    # 2. Calculate relative coordinates BEFORE clipping/padding
    # These are the coordinates relative to the 130x130 frame
    rx = x - start_x
    rx2 = x2 - start_x
    ry = y - start_y
    ry2 = y2 - start_y
    
    # 3. Handle Slicing and Padding (same as before)
    end_x, end_y = start_x + crop_size, start_y + crop_size
    
    pad_left = max(0, -start_x)
    pad_top = max(0, -start_y)
    pad_right = max(0, end_x - array.shape[1])
    pad_bottom = max(0, end_y - array.shape[0])
    
    slice_x1, slice_x2 = max(0, start_x), min(array.shape[1], end_x)
    slice_y1, slice_y2 = max(0, start_y), min(array.shape[0], end_y)
    
    crop = array[slice_y1:slice_y2, slice_x1:slice_x2]
    
    if pad_left > 0 or pad_right > 0 or pad_top > 0 or pad_bottom > 0:
        padding = [(pad_top, pad_bottom), (pad_left, pad_right)]
        if array.ndim == 3: padding.append((0, 0))
        crop = np.pad(crop, padding, mode='constant')
        
    return crop, Bbox(rx, ry, rx2-rx, ry2-ry)
    # return crop, (rx, rx2, ry, ry2)

# to use: B5P5K5S3
def get_blurred_artifacts(
    img, bbox: Bbox, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    assert img is not None
    new_mask, box = get_padded_bbox_mask(img.shape, bbox, bbox_pad)

    blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
    only_mask_unblurred = blurred.copy()

    new_mask = new_mask.astype(bool)
    only_mask_unblurred[new_mask] = img[new_mask]

    return {
        "padded_mask": new_mask,
        "unblurred": only_mask_unblurred,
        "img": img,
    }


def get_padded_bbox_mask(shape, bbox: Bbox, padding=5):
    # 2. Get the standard bounding box
    img_h, img_w = shape[:2]

    # 3. Apply padding with boundary constraints
    x1 = max(0, bbox.x - padding)
    y1 = max(0, bbox.y - padding)
    x2 = min(img_w, bbox.x2 + padding)
    y2 = min(img_h, bbox.y2 + padding)

    # 4. Create the new mask
    padded_mask = np.zeros(shape, dtype=np.uint8)
    cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

    return padded_mask, (x1, y1, x2, y2)



In [ ]:
p = learn.dls.valid_ds.items[2]
d, _ = get_originals(p)
image, mask = DiskImage.load(d / "orig.jpg"), DiskBooleanMask.load(d / "mask.png")

predict_and_return_prob_masks(image, mask, learn)

show([image, mask])

In [ ]:
TEST_DS = [
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/532628267735038"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1765386860738037"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/167994091824213"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2985603351766082"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2800723576860190"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/871543466734475"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2562170430756718"
    ),
]

In [ ]:
image, m2 = DiskImage.load(TEST_DS[0] / "image.jpg"), DiskBooleanMask.load(TEST_DS[0] / "m2.png")
unblurred = predict_and_return_prob_masks(image, m2, learn)

In [ ]:
unblurred[0].shape

In [ ]:
show(unblurred[:4])

In [ ]:
from collections import Counter

tc = Counter([label_func(i.name) for i in dls.train_ds.items])
vc = Counter([label_func(i.name) for i in dls.valid_ds.items])

tc, vc

In [ ]:
(4807 + 1976) / 1976, (4807 + 1976) / 4807

In [ ]:
from torch import nn

learn = vision_learner(
    dls,
    resnet18,
    metrics=[F1Score(average="macro"), Precision(), Recall()],
    loss_func=CrossEntropyLossFlat(weight=torch.tensor([1.41, 3.43])),
)

# 2. Modify the 'stem' (first layer)
# Change kernel to 3x3 and stride to 1 to preserve detail
learn.model[0][0] = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# 3. Remove the initial MaxPool (set to Identity)
# Standard ResNet uses MaxPool immediately after the first conv;
# for small images, this "blurs" your small area of focus.
learn.model[0][3] = nn.Identity()
learn = learn.remove_cb(ProgressCallback)

In [ ]:
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(5)

In [ ]:
learn.show_results()

In [ ]:
! say "jupyter finish"

In [ ]:
learn.export("unblurred_stage1_0pad_crops_b5p5_resnet18.pkl")

# Stage 2
Train on unblurred dataset

In [ ]:
# learner = load_learner(LOG_ROOT / "unblurred_stage1_0pad_crops_b5p5_resnet18.pkl")

In [ ]:
BLURRED_ROOT = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred"
)
B5P5_BLUR_K5S3 = BLURRED_ROOT / "b5p5_blur_k5s3"

In [ ]:
dls = ImageDataLoaders.from_name_func(
    LOG_ROOT,
    get_image_files(B5P5_BLUR_K5S3),
    valid_pct=0.2,
    seed=42,
    label_func=label_func,
    item_tfms=CropPad(130),
    batch_tfms=aug_transforms(max_zoom=1.0),
    bs=8,
)


In [ ]:
dls.show_batch()

In [ ]:
learn.dls = dls

In [ ]:
learn.freeze()

In [ ]:
learn.show_results()

In [ ]:
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(5)

In [ ]:
# learn.export
learn.export("unblurred_stage2_blurpad_crops_b5p5k5s3_resnet18.pkl")

In [ ]:
learn.lr_find()

# what is not working?